# 🧠 NeuroScan-AI — YOLO Brain Tumor Detection
## Kaggle Training Notebook
- Dataset: BraTS 2021
- Model: YOLOv8n (Pretrained)
- Task: Tumor Localization (Bounding Box)
- Classes: 3 (Necrotic, Edema, Enhancing Tumor)
- Tracking: MLflow + Dagshub
- Training: Kaggle GPU T4 x2

In [25]:
!pip install ultralytics dagshub mlflow nibabel opencv-python-headless -q

In [26]:
import os
print(os.listdir('/kaggle/input/datasets/dschettler8845/brats-2021-task1'))

['BraTS2021_00495.tar', 'BraTS2021_Training_Data.tar', 'BraTS2021_00621.tar']


In [27]:
import os
import mlflow
from kaggle_secrets import UserSecretsClient

# Dagshub + MLflow connect — NO browser auth
secrets = UserSecretsClient()
dagshub_token = secrets.get_secret("DAGSHUB_TOKEN")

os.environ["MLFLOW_TRACKING_USERNAME"] = "kaushik-chariya"
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow"

mlflow.set_tracking_uri("https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow")
mlflow.set_experiment("YOLO-Brain-Tumor-Detection")

print("✅ Connected! https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow")

✅ Connected! https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow


In [28]:
import tarfile
import os

tar_path = '/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar'
extract_path = '/kaggle/working/BraTS2021'

os.makedirs(extract_path, exist_ok=True)

print("⏳ Extracting...")
with tarfile.open(tar_path, 'r') as tar:
    tar.extractall(extract_path)

print("✅ Done!")
print(os.listdir(extract_path)[:3])

⏳ Extracting...


/tmp/ipykernel_58/4118965996.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_path)


✅ Done!
['BraTS2021_01484', 'BraTS2021_00588', 'BraTS2021_00016']


In [29]:
patients = sorted([
    p for p in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, p))
])
print(f"✅ Total patients: {len(patients)}")
print(f"✅ Sample: {patients[:3]}")

✅ Total patients: 1251
✅ Sample: ['BraTS2021_00000', 'BraTS2021_00002', 'BraTS2021_00003']


In [30]:
def get_bounding_boxes(seg_slice, image_size=640):
    boxes = []
    h, w = seg_slice.shape
    
    for class_id, class_val in [(0, 1), (1, 2), (2, 4)]:
        mask = (seg_slice == class_val).astype(np.uint8)
        
        if mask.sum() == 0:
            continue
        
        # Merge all contours into ONE bounding box per class
        coords = np.where(mask > 0)
        if len(coords[0]) == 0:
            continue
            
        y_min, y_max = coords[0].min(), coords[0].max()
        x_min, x_max = coords[1].min(), coords[1].max()
        
        bw = x_max - x_min
        bh = y_max - y_min
        
        # Better minimum size threshold
        if bw < 10 or bh < 10:
            continue
        
        # Normalize to YOLO format
        x_center = (x_min + bw / 2) / w
        y_center = (y_min + bh / 2) / h
        norm_w = bw / w
        norm_h = bh / h
        
        boxes.append([class_id, x_center, y_center, norm_w, norm_h])
    
    return boxes

print("✅ Bounding box function ready!")

✅ Bounding box function ready!


In [31]:
def prepare_yolo_dataset(patients, dataset_path, output_path, slice_indices=list(range(40, 150, 3))):  # ← only this changed
    """
    BraTS data to YOLO format converter.
    """
    images_dir = os.path.join(output_path, "images")
    labels_dir = os.path.join(output_path, "labels")
    
    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)
    
    total_saved = 0
    skipped = 0
    
    def norm(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-8)
    
    for i, patient in enumerate(patients):
        patient_path = os.path.join(dataset_path, patient)
        
        try:
            files = os.listdir(patient_path)
            
            t1ce_f  = [f for f in files if "t1ce" in f and f.endswith(".nii.gz")]
            t2_f    = [f for f in files if "t2" in f and f.endswith(".nii.gz")]
            flair_f = [f for f in files if "flair" in f and f.endswith(".nii.gz")]
            seg_f   = [f for f in files if "seg" in f and f.endswith(".nii.gz")]
            
            if not all([t1ce_f, t2_f, flair_f, seg_f]):
                skipped += 1
                continue
            
            t1ce_vol  = nib.load(os.path.join(patient_path, t1ce_f[0])).get_fdata()
            t2_vol    = nib.load(os.path.join(patient_path, t2_f[0])).get_fdata()
            flair_vol = nib.load(os.path.join(patient_path, flair_f[0])).get_fdata()
            seg_vol   = nib.load(os.path.join(patient_path, seg_f[0])).get_fdata()
            
            t1ce_vol  = norm(t1ce_vol)
            t2_vol    = norm(t2_vol)
            flair_vol = norm(flair_vol)
            
            for idx in slice_indices:
                if idx >= seg_vol.shape[2]:
                    continue
                
                seg_slice = seg_vol[:, :, idx]
                boxes = get_bounding_boxes(seg_slice)
                
                if not boxes:
                    continue
                
                t1ce_s  = t1ce_vol[:, :, idx]
                t2_s    = t2_vol[:, :, idx]
                flair_s = flair_vol[:, :, idx]
                
                img = np.stack([t1ce_s, t2_s, flair_s], axis=-1)
                img = (img * 255).astype(np.uint8)
                img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
                
                fname = f"{patient}_slice{idx}"
                cv2.imwrite(os.path.join(images_dir, f"{fname}.jpg"), img)
                
                with open(os.path.join(labels_dir, f"{fname}.txt"), "w") as f:
                    for box in boxes:
                        f.write(" ".join(map(str, box)) + "\n")
                
                total_saved += 1
        
        except Exception as e:
            print(f"⚠️ Skipping {patient}: {e}")
            skipped += 1
            continue
        
        if (i + 1) % 100 == 0:
            print(f"   Processed {i+1}/{len(patients)}")
    
    print(f"\n✅ Total images saved: {total_saved}")
    print(f"⚠️  Skipped: {skipped}")
    return total_saved

# Run
OUTPUT_PATH = "/kaggle/working/yolo_dataset"
total = prepare_yolo_dataset(patients, DATASET_PATH, OUTPUT_PATH)

   Processed 100/1251
   Processed 200/1251
   Processed 300/1251
   Processed 400/1251
   Processed 500/1251
   Processed 600/1251
   Processed 700/1251
   Processed 800/1251
   Processed 900/1251
   Processed 1000/1251
   Processed 1100/1251
   Processed 1200/1251

✅ Total images saved: 24185
⚠️  Skipped: 0


In [32]:
import shutil
import random

random.seed(42)

# Paths
images = sorted(os.listdir(os.path.join(OUTPUT_PATH, "images")))
random.shuffle(images)

# 80/20 split
split = int(0.8 * len(images))
train_imgs = images[:split]
val_imgs   = images[split:]

# Folders banao
for split_name in ["train", "val"]:
    os.makedirs(f"{OUTPUT_PATH}/{split_name}/images", exist_ok=True)
    os.makedirs(f"{OUTPUT_PATH}/{split_name}/labels", exist_ok=True)

# Copy files
for img in train_imgs:
    label = img.replace(".jpg", ".txt")
    shutil.copy(f"{OUTPUT_PATH}/images/{img}",   f"{OUTPUT_PATH}/train/images/{img}")
    shutil.copy(f"{OUTPUT_PATH}/labels/{label}", f"{OUTPUT_PATH}/train/labels/{label}")

for img in val_imgs:
    label = img.replace(".jpg", ".txt")
    shutil.copy(f"{OUTPUT_PATH}/images/{img}",   f"{OUTPUT_PATH}/val/images/{img}")
    shutil.copy(f"{OUTPUT_PATH}/labels/{label}", f"{OUTPUT_PATH}/val/labels/{label}")

print(f"✅ Train: {len(train_imgs)} images")
print(f"✅ Val  : {len(val_imgs)} images")

✅ Train: 26563 images
✅ Val  : 6641 images


In [33]:
# data.yaml banao
yaml_content = f"""
path: {OUTPUT_PATH}
train: train/images
val: val/images

nc: 3
names:
  0: necrotic
  1: edema
  2: enhancing_tumor
"""

yaml_path = "/kaggle/working/data.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("✅ data.yaml created!")
print(yaml_content)


✅ data.yaml created!

path: /kaggle/working/yolo_dataset
train: train/images
val: val/images

nc: 3
names:
  0: necrotic
  1: edema
  2: enhancing_tumor



In [ ]:
from ultralytics import YOLO

with mlflow.start_run(run_name="YOLO-BraTS2021-V4-Industry"):
    
    mlflow.log_params({
        "model":              "yolov8x",
        "epochs":             100,
        "batch":              8,
        "image_size":         640,
        "dataset":            "BraTS2021",
        "train_samples":      len(train_imgs),
        "val_samples":        len(val_imgs),
        "classes":            "necrotic, edema, enhancing_tumor",
        "modalities":         "T1ce+T2+FLAIR",
        "conf_thresh":        0.25,
        "iou_thresh":         0.5,
        "slices_per_patient": 37
    })
    
    model = YOLO("yolov8x.pt")
    
    results = model.train(
        data=yaml_path,
        epochs=100,
        batch=8,
        imgsz=640,
        device="0",
        workers=4,
        cache=True,
        
        # Optimizer
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=5.0,
        
        # Augmentation
        mosaic=0.5,
        fliplr=0.5,
        flipud=0.5,
        degrees=15.0,
        scale=0.3,
        hsv_h=0.0,
        hsv_s=0.3,
        hsv_v=0.3,
        
        # Accuracy
        conf=0.25,
        iou=0.5,
        patience=30,
        
        project="/kaggle/working/yolo_runs",
        name="brain_tumor_industry",
        exist_ok=True,
        verbose=True
    )
    
    metrics = results.results_dict
    mlflow.log_metrics({
        "mAP50":     metrics.get("metrics/mAP50(B)", 0),
        "mAP50_95":  metrics.get("metrics/mAP50-95(B)", 0),
        "precision": metrics.get("metrics/precision(B)", 0),
        "recall":    metrics.get("metrics/recall(B)", 0),
    })
    
    best_model_path = "/kaggle/working/yolo_runs/brain_tumor_industry/weights/best.pt"
    final_path      = "/kaggle/working/yolo_model.pt"
    
    shutil.copy(best_model_path, final_path)
    mlflow.log_artifact(final_path)
    
    print(f"\n✅ mAP50    : {metrics.get('metrics/mAP50(B)', 0):.4f}")
    print(f"✅ mAP50-95 : {metrics.get('metrics/mAP50-95(B)', 0):.4f}")
    print(f"✅ Precision: {metrics.get('metrics/precision(B)', 0):.4f}")
    print(f"✅ Recall   : {metrics.get('metrics/recall(B)', 0):.4f}")
    print("✅ YOLO Model saved + logged to MLflow!")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=0.25, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.3, hsv_v=0.3, imgsz=640, int8=False, iou=0.5, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=brain_tumor_industry, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patien

2026/06/05 08:05:22 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/06/05 08:05:22 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(b6be585dbc474e3f8bf98a8a8f207ec8) to https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow
MLflow: disable with 'yolo settings mlflow=False'
WARNING ⚠️ MLflow: Failed to initialize: INVALID_PARAMETER_VALUE: Response: {'error_code': 'INVALID_PARAMETER_VALUE'}
WARNING ⚠️ MLflow: Not tracking this run
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /kaggle/working/yolo_runs/brain_tumor_industry
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      6.95G      1.672      1.757      1.758          8        640: 9% ━─────────── 309/3597 1.3it/s 3:19<42:39